**Celda** 1: Montar Drive y crear carpetas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
BASE = '/content/drive/MyDrive/chess_coach'
os.makedirs(f'{BASE}/data', exist_ok=True)
os.makedirs(f'{BASE}/data/pgn_games', exist_ok=True)
os.makedirs(f'{BASE}/engines', exist_ok=True)
sys.path.append(BASE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Celda 2: Instalar dependencias

In [ ]:
!pip install chess -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 50.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


Celda 3: Descargar e instalar Stockfish

In [ ]:
!curl -s https://api.github.com/repos/official-stockfish/Stockfish/releases/latest | grep "browser_download_url.*ubuntu-x86-64-avx2.tar" | cut -d '"' -f4 | xargs wget -O /content/stockfish.tar
!tar -xf /content/stockfish.tar -C /content
!chmod +x /content/stockfish/stockfish-ubuntu-x86-64-avx2
!/content/stockfish/stockfish-ubuntu-x86-64-avx2 --help

--2026-08-08 17:19:24--  https://github.com/official-stockfish/Stockfish/releases/download/sf_18/stockfish-ubuntu-x86-64-avx2.tar
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/20976138/f9258687-6455-4259-8dab-22a0e0dc4295?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-08-08T18%3A01%3A17Z&rscd=attachment%3B+filename%3Dstockfish-ubuntu-x86-64-avx2.tar&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-08-08T17%3A00%3A38Z&ske=2026-08-08T18%3A01%3A17Z&sks=b&skv=2018-11-09&sig=E6wUzVxO8tpCMMSlcjEfjEsNXp2O1mdDvVsoep80OSg%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4NjIxMzE2NCwibmJmIjoxNzg2MjA5NTY0LCJwY

Celda 4: Crear translations.py

In [ ]:
%%writefile /content/drive/MyDrive/chess_coach/translations.py

TEXT = {
    "en": {
        "welcome": "Welcome to the AI Chess Club Coach!",
        "choose_option": "Please choose an option:",
        "menu_1": "1. Personalized Training Plan",
        "menu_2": "2. Opening Recommender",
        "menu_3": "3. Blunder Tracker",
        "menu_0": "0. Exit",
        "invalid": "Invalid choice. Please try again.",

        "ask_elo": "What is your current Elo rating? ",
        "ask_style": "Preferred style (1=Aggressive, 2=Defensive, 3=Strategic): ",
        "ask_time": "How many days per week can you study? ",
        "plan_saved": "Plan saved at:",
        "modules": {
            "beginner":     ["Basic Tactics", "Mating Patterns in 1", "Rules and Notation"],
            "intermediate": ["Intermediate Tactics", "Pawn Endgames", "Basic Openings"],
            "advanced":     ["Endgame Principles", "Pawn Structures", "Strategic Planning"],
        },
        "style_extra": {
            "1": "Study of sacrifices and king attacks",
            "2": "Study of fortresses and passive defense",
            "3": "Study of long-term plans",
        },

        "opening_style_prompt": "Preferred style (1=Sharp, 2=Controlled, 3=Balanced): ",
        "openings": {
            "sharp":      {"name": "Sicilian Defense",  "moves": ["e4", "c5", "Nf3", "d6", "d4", "cxd4", "Nxd4", "Nf6"]},
            "controlled": {"name": "Queen's Gambit",    "moves": ["d4", "d5", "c4", "e6", "Nc3", "Nf6", "Nf3", "Be7"]},
            "balanced":   {"name": "Italian Game",      "moves": ["e4", "e5", "Nf3", "Nc6", "Bc4", "Bc5", "c3", "Nf6"]},
        },
        "training_label": "Training:",
        "move_instruction": "Type the move in SAN notation (e.g. e4, Nf3)",
        "move_prompt": "Move {num} ({turn}), expected: ",
        "white": "White",
        "black": "Black",
        "correct": "✅ Correct!\n",
        "incorrect": "❌ Expected: {move}\n",
        "training_end": "End of opening training.",

        "pgn_prompt": "How do you want to provide the PGN?",
        "pgn_option1": "1. Paste the text directly",
        "pgn_option2": "2. Auto-detect file in Drive",
        "file_detected": "File detected:",
        "paste_pgn": "Paste your PGN:\n",
        "blunder_found": "Blunder detected on move {move_num}! Loss of evaluation: {drop}",
        "no_blunders": "No blunders detected.",
        "stockfish_missing": "⚠️ Stockfish is not available in this session.\nRun the Stockfish download cells (wget + tar + chmod) and try again.\n",
        "file_not_found": "⚠️ File not found:",
    },
    "es": {
        "welcome": "¡Bienvenido al Entrenador de Club de Ajedrez con IA!",
        "choose_option": "Por favor, elige una opción:",
        "menu_1": "1. Plan de Entrenamiento Personalizado",
        "menu_2": "2. Recomendador de Aperturas",
        "menu_3": "3. Rastreador de Errores Graves",
        "menu_0": "0. Salir",
        "invalid": "Opción no válida. Por favor, inténtalo de nuevo.",

        "ask_elo": "¿Cuál es tu Elo actual? ",
        "ask_style": "Estilo preferido (1=Agresivo, 2=Defensivo, 3=Estratégico): ",
        "ask_time": "¿Cuántos días a la semana puedes estudiar? ",
        "plan_saved": "Plan guardado en:",
        "modules": {
            "beginner":     ["Tácticas Básicas", "Patrones de Mate en 1", "Reglas y Notación"],
            "intermediate": ["Tácticas Intermedias", "Finales de Peones", "Aperturas Básicas"],
            "advanced":     ["Principios de Finales", "Estructuras de Peones", "Planificación Estratégica"],
        },
        "style_extra": {
            "1": "Estudio de sacrificios y ataques al rey",
            "2": "Estudio de fortalezas y defensa pasiva",
            "3": "Estudio de planes a largo plazo",
        },

        "opening_style_prompt": "Estilo preferido (1=Afiladas, 2=Controladas, 3=Equilibradas): ",
        "openings": {
            "sharp":      {"name": "Defensa Siciliana",   "moves": ["e4", "c5", "Nf3", "d6", "d4", "cxd4", "Nxd4", "Nf6"]},
            "controlled": {"name": "Gambito de Dama",     "moves": ["d4", "d5", "c4", "e6", "Nc3", "Nf6", "Nf3", "Be7"]},
            "balanced":   {"name": "Apertura Italiana",   "moves": ["e4", "e5", "Nf3", "Nc6", "Bc4", "Bc5", "c3", "Nf6"]},
        },
        "training_label": "Entrenando:",
        "move_instruction": "Escribe el movimiento en notación SAN (ej: e4, Nf3)",
        "move_prompt": "Jugada {num} ({turn}), correcta esperada: ",
        "white": "Blancas",
        "black": "Negras",
        "correct": "✅ ¡Correcto!\n",
        "incorrect": "❌ Se esperaba: {move}\n",
        "training_end": "Fin del entrenamiento de apertura.",

        "pgn_prompt": "¿Cómo quieres dar el PGN?",
        "pgn_option1": "1. Pegar el texto directamente",
        "pgn_option2": "2. Detectar automáticamente el archivo en Drive",
        "file_detected": "Archivo detectado:",
        "paste_pgn": "Pega tu PGN:\n",
        "blunder_found": "¡Error grave detectado en la jugada {move_num}! Pérdida de evaluación: {drop}",
        "no_blunders": "No se detectaron errores graves.",
        "stockfish_missing": "⚠️ Stockfish no está disponible en esta sesión.\nCorre las celdas de descarga de Stockfish (wget + tar + chmod) y vuelve a intentar.\n",
        "file_not_found": "⚠️ No se encontró el archivo:",
    }
}

Overwriting /content/drive/MyDrive/chess_coach/translations.py


Celda 5: Crear training_planner.py

In [ ]:
%%writefile /content/drive/MyDrive/chess_coach/training_planner.py

import json
import os
from datetime import date, timedelta

def get_level(elo: int) -> str:
    if elo < 1000:
        return "beginner"
    elif elo < 1500:
        return "intermediate"
    else:
        return "advanced"

def generate_schedule(elo: int, style: str, days_per_week: int, lang_dict: dict) -> list:
    level = get_level(elo)
    modules = lang_dict["modules"][level] + [lang_dict["style_extra"].get(style, "")]
    schedule = []
    today = date.today()
    for i in range(days_per_week * 4):
        day = today + timedelta(days=i)
        module = modules[i % len(modules)]
        schedule.append({"date": str(day), "module": module, "completed": False})
    return schedule

def save_progress(schedule: list, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(schedule, f, ensure_ascii=False, indent=2)

def load_progress(path: str) -> list:
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return []

Overwriting /content/drive/MyDrive/chess_coach/training_planner.py


Celda 6: Crear opening_recommender.py

In [ ]:
%%writefile /content/drive/MyDrive/chess_coach/opening_recommender.py

import chess

def recommend(preference: str, lang_dict: dict) -> dict:
    return lang_dict["openings"].get(preference, lang_dict["openings"]["balanced"])

def train_opening(opening: dict, lang_dict: dict):
    board = chess.Board()
    print(f"\n{lang_dict['training_label']} {opening['name']}")
    print(lang_dict["move_instruction"] + "\n")

    for i, correct_move in enumerate(opening["moves"]):
        print(board)
        turn = lang_dict["white"] if i % 2 == 0 else lang_dict["black"]
        user_move = input("\n" + lang_dict["move_prompt"].format(num=i+1, turn=turn))

        if user_move.strip() == correct_move:
            print(lang_dict["correct"])
        else:
            print(lang_dict["incorrect"].format(move=correct_move))

        board.push_san(correct_move)

    print(lang_dict["training_end"])

Overwriting /content/drive/MyDrive/chess_coach/opening_recommender.py


Celda 7: Crear blunder_tracker.py

In [ ]:
%%writefile /content/drive/MyDrive/chess_coach/blunder_tracker.py

import chess
import chess.pgn
import chess.engine
import io
import os
import glob

STOCKFISH_PATH = "/content/stockfish/stockfish-ubuntu-x86-64-avx2"
BLUNDER_THRESHOLD = 2.0

def analyze_pgn(pgn_text: str) -> list:
    game = chess.pgn.read_game(io.StringIO(pgn_text))
    board = game.board()
    blunders = []

    with chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH) as engine:
        prev_score = None
        move_num = 1
        for move in game.mainline_moves():
            board.push(move)
            info = engine.analyse(board, chess.engine.Limit(depth=12))
            score = info["score"].white().score(mate_score=10000)
            if score is not None and prev_score is not None:
                drop = abs(prev_score - score) / 100.0
                if drop >= BLUNDER_THRESHOLD:
                    blunders.append({"move_num": move_num, "move": board.peek().uci(), "drop": round(drop, 2)})
            prev_score = score
            if board.turn == chess.WHITE:
                move_num += 1

    return blunders

def read_pgn_from_file(filepath: str) -> str:
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"No se encontró el archivo: {filepath}")
    with open(filepath, "r", encoding="utf-8") as f:
        return f.read()

def find_latest_pgn(folder: str) -> str:
    archivos = glob.glob(os.path.join(folder, "*.pgn"))
    if not archivos:
        raise FileNotFoundError(f"No se encontró ningún archivo .pgn en: {folder}")
    return max(archivos, key=os.path.getmtime)

Overwriting /content/drive/MyDrive/chess_coach/blunder_tracker.py


Celda 8: Crear main.py (ahora todo pasa por t[...])

In [ ]:
%%writefile /content/drive/MyDrive/chess_coach/main.py

import os
from translations import TEXT
import training_planner
import opening_recommender
import blunder_tracker

STOCKFISH_PATH = "/content/stockfish/stockfish-ubuntu-x86-64-avx2"
PGN_FOLDER = "/content/drive/MyDrive/chess_coach/data/pgn_games"

def main():
    print("Select Language / Seleccione el Idioma:")
    print("1. English")
    print("2. Español")
    lang = "es" if input("> ") == "2" else "en"
    t = TEXT[lang]

    print("\n" + "=" * 40)
    print(t["welcome"])
    print("=" * 40)

    while True:
        print(f"\n{t['choose_option']}")
        print(t["menu_1"])
        print(t["menu_2"])
        print(t["menu_3"])
        print(t["menu_0"])
        choice = input("> ")

        if choice == "1":
            elo = int(input(t["ask_elo"]))
            style = input(t["ask_style"])
            days = int(input(t["ask_time"]))
            plan = training_planner.generate_schedule(elo, style, days, t)
            for entry in plan[:7]:
                print(f"{entry['date']} → {entry['module']}")
            path = "/content/drive/MyDrive/chess_coach/data/progress.json"
            training_planner.save_progress(plan, path)
            print("\n" + t["plan_saved"], path)

        elif choice == "2":
            pref = input(t["opening_style_prompt"])
            key = {"1": "sharp", "2": "controlled", "3": "balanced"}.get(pref, "balanced")
            opening = opening_recommender.recommend(key, t)
            opening_recommender.train_opening(opening, t)

        elif choice == "3":
            if not os.path.exists(STOCKFISH_PATH):
                print("\n" + t["stockfish_missing"])
                continue

            print("\n" + t["pgn_prompt"])
            print(t["pgn_option1"])
            print(t["pgn_option2"])
            modo = input("> ")

            if modo == "2":
                try:
                    ruta = blunder_tracker.find_latest_pgn(PGN_FOLDER)
                    print(f"{t['file_detected']} {ruta}")
                    pgn_text = blunder_tracker.read_pgn_from_file(ruta)
                except FileNotFoundError:
                    print(f"{t['file_not_found']} {PGN_FOLDER}")
                    continue
            else:
                pgn_text = input(t["paste_pgn"])

            blunders = blunder_tracker.analyze_pgn(pgn_text)
            if blunders:
                for b in blunders:
                    print(t["blunder_found"].format(move_num=b["move_num"], drop=b["drop"]))
            else:
                print(t["no_blunders"])

        elif choice == "0":
            break
        else:
            print(t["invalid"])

if __name__ == "__main__":
    main()

Writing /content/drive/MyDrive/chess_coach/main.py


Celda 9: Ejecutar el programa

In [ ]:
!python /content/drive/MyDrive/chess_coach/main.py

Select Language / Seleccione el Idioma:
1. English
2. Español
> 2

¡Bienvenido al Entrenador de Club de Ajedrez con IA!

Por favor, elige una opción:
1. Plan de Entrenamiento Personalizado
2. Recomendador de Aperturas
3. Rastreador de Errores Graves
0. Salir
> 3

¿Cómo quieres dar el PGN?
1. Pegar el texto directamente
2. Detectar automáticamente el archivo en Drive
> 2
Archivo detectado: /content/drive/MyDrive/chess_coach/data/pgn_games/Perugoalie_vs_Jaggi1729_2026.08.05.pgn
¡Error grave detectado en la jugada 18! Pérdida de evaluación: 3.32
¡Error grave detectado en la jugada 19! Pérdida de evaluación: 5.04
¡Error grave detectado en la jugada 26! Pérdida de evaluación: 2.0
¡Error grave detectado en la jugada 27! Pérdida de evaluación: 3.04
¡Error grave detectado en la jugada 28! Pérdida de evaluación: 3.34
¡Error grave detectado en la jugada 29! Pérdida de evaluación: 3.23
¡Error grave detectado en la jugada 34! Pérdida de evaluación: 2.06
¡Error grave detectado en la jugada 37! Pérd